# Polymorphism Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. One loop, many voices.** Each class supplies its own `speak()`; the caller never checks types, so new species cost zero loop edits.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."


class Dog(Animal):
    def speak(self):
        return f"{self.name}: Woof!"


class Cat(Animal):
    def speak(self):
        return f"{self.name}: Meow."


farm = [Dog("Rex"), Cat("Manik")]
for animal in farm:
    print(animal.speak())          # ONE calling line, many behaviors


class Duck(Animal):                # brand-new species...
    def speak(self):
        return f"{self.name}: Quack!"


farm.append(Duck("Khaki"))
for animal in farm:
    print(animal.speak())          # ...zero loop edits needed

**2. Built-ins do it too.** `len()` dispatches to each type's own `__len__` — you have been using polymorphism since week one.

In [ ]:
print(len("Dhaka"))                    # str: counts characters
print(len([10, 20, 30]))               # list: counts elements
print(len({"a": 1, "b": 2, "c": 3}))   # dict: counts keys
print(len(range(100)))                 # range: length without building anything

print("Dhaka".__len__(), [10, 20, 30].__len__())
# One name, many implementations - the whole language is polymorphic.

**3. Operators join in.** `+` means arithmetic, concatenation or merging depending on the operands — operator overloading in the wild.

In [ ]:
print(7 + 5)                  # ints: arithmetic
print("poly" + "morphism")    # strs: concatenation
print([1, 2] + [3])           # lists: merging
print(3 * "ab")               # bonus: '*' repeats sequences too
# Four behaviors, one familiar symbol.

## Part 2 — Practice

**4. Strangers that quack.** Duck typing needs no shared ancestor — Python only checks that the method exists at CALL time.

In [ ]:
class Duck:
    def quack(self):
        return "Quack!"


class Robot:
    def quack(self):
        return "Beep-quack module online."


class Prankster:
    def quack(self):
        return "(it is a human with a reed)"


for candidate in (Duck(), Robot(), Prankster()):   # no shared parent,
    print(candidate.quack())                       # nor does it matter


class SilentStone:
    pass                                           # no quack() anywhere


pond = [Duck(), SilentStone()]
for candidate in pond:
    try:
        print(candidate.quack())
    except AttributeError as err:
        print("AttributeError:", err)

**5. Teach `+` about money.** `lunch + coffee` dispatches to `Money.__add__`, which builds a fresh object instead of mutating either side.

In [ ]:
class Money:
    """An amount in BDT that knows how to add itself."""

    def __init__(self, amount):
        self.amount = amount

    def __add__(self, other):           # teaches '+' what Money + Money means
        return Money(self.amount + other.amount)

    def __repr__(self):
        return f"Money({self.amount})"


lunch = Money(250)
coffee = Money(80)

total = lunch + coffee
print(total)
print(total + Money(20))

**6. The shape contract.** Raising `NotImplementedError` turns polite convention into loud failure the moment a subclass forgets its duty.

In [ ]:
class Shape:
    """Contract: every shape MUST know its area."""

    def area(self):
        raise NotImplementedError("subclass must implement area()")


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return round(3.14159 * self.radius ** 2, 2)


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2


for s in (Circle(3), Square(4)):
    print(type(s).__name__, "area =", s.area())

try:
    Shape().area()                     # the contract, left unfilled
except NotImplementedError as err:
    print("NotImplementedError:", err)

**7. Retire the if/else chain.** Type-branching scatters one decision across every call site; polymorphism moves each decision into its own class.

In [ ]:
class Dog:
    pass


class Cat:
    pass


class Cow:
    pass


def make_sound(animal):
    kind = type(animal).__name__
    if kind == "Dog":
        return "Woof!"
    elif kind == "Cat":
        return "Meow."
    else:
        raise ValueError(f"I don't know a {kind}")


print(make_sound(Dog()))
print(make_sound(Cat()))
try:
    make_sound(Cow())                  # every new species breaks every branch
except ValueError as err:
    print("ValueError:", err)


# Polymorphic rewrite: the decision moves INTO each class.
class Dog:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name}: Woof!"


class Cat:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name}: Meow."


class Cow:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name}: Moo!"


for animal in (Dog("Rex"), Cat("Manik"), Cow("Lali")):
    print(animal.speak())
# A new species now costs one small class - zero edits anywhere else.

## Part 3 — Challenge

**8. Uniform checkout.** Production polymorphism: payment providers come and go, checkout never notices.

In [ ]:
class CardPayment:
    def __init__(self, last4):
        self.last4 = last4

    def pay(self, amount):
        return f"Card ****{self.last4}: paid {amount} BDT"


class PayPalPayment:
    def __init__(self, email):
        self.email = email

    def pay(self, amount):
        return f"PayPal <{self.email}>: paid {amount} BDT"


class WalletPayment:
    def __init__(self, provider):
        self.provider = provider

    def pay(self, amount):
        return f"{self.provider} wallet: paid {amount} BDT"


def checkout(total, methods):
    """Process one total through EVERY payment method uniformly."""
    print(f"-- invoice: {total} BDT --")
    for method in methods:
        print(" ", method.pay(total))


checkout(3499, [CardPayment("4242"),
                PayPalPayment("sarah@example.com"),
                WalletPayment("bKash")])


class CryptoPayment:                   # drops in years later...
    def pay(self, amount):
        return f"Crypto: paid {amount} BDT"


checkout(199, [CryptoPayment()])       # ...checkout unchanged

**9. Broadcast with a contract.** The base-class `raise` documents the interface and turns a forgotten implementation into an immediate, obvious failure.

In [ ]:
class Notifier:
    """Contract: every channel MUST implement send()."""

    def send(self, message):
        raise NotImplementedError("subclass must implement send()")


class EmailNotifier(Notifier):
    def send(self, message):
        return f"[email] {message}"


class SmsNotifier(Notifier):
    def send(self, message):
        return f"[sms] {message}"


class PushNotifier(Notifier):
    def send(self, message):
        return f"[push] {message}"


def broadcast(message, channels):
    for channel in channels:           # duck typing in production clothes
        print(channel.send(message))


broadcast("deploy finished", [EmailNotifier(), SmsNotifier(), PushNotifier()])


class LazyNotifier(Notifier):
    pass                               # forgot send()


try:
    broadcast("will this work?", [LazyNotifier()])
except NotImplementedError as err:
    print("NotImplementedError:", err)